# 첫번째와 두 번째 셀에 있는 crop_name 과 item_code를 바꾸고 사용하시면 됩니다.
## 결측치가 있는 행은 제거 됩니다. (ex: 2018년 1~4주차는 lag 데이터가 없어서 제거됨)
### 아이템 코드
- 양파 : 1201
- 배추 : 1001
- 상추 : 1005
- 사과 : 0601
- 무 : 1101
- 감자 : 0501
- 대파 : 1202
- 건고추 : 1207
- 마늘 : 1209
- 딸기 : 0804
- 방울토마토 : 0806
- 오이 : 0901
- 양배추 : 1004
- 고구마 : 0502
- 배 : 0602

In [199]:
import pandas as pd

# ✅ 작물 이름 바꾸기
crop_name = '상추'  # 여기만 '사과', '무', '양파' 등으로 바꾸면 됨

# ✅ 파일 경로 자동 생성
file_path = f"../data/{crop_name}_이상치제거_주간기준_등급코드.csv"

# 파일 로드
df = pd.read_csv(file_path, encoding='cp949')
df_grow = pd.read_csv('../data/factor_external_weekly.csv', encoding='utf-8') # 이건 고정
# 이후 df를 기반으로 전처리, 병합, lag 생성 등 전체 코드 실행


In [200]:
# ✅ 작물 코드 바꾸기
df['item_code'] = '1005'  # 거래데이터에는 대분류 코드가 없고 한 종류만 있음

df_grow['item_code'] = df_grow['item_code'].astype(str) 
df_grow['item_code'] = df_grow['item_code'].str.zfill(4)

In [201]:
# weekno 열을 만들고 주차 표기를 통일시켜 merge 준비
def get_week_of_year(date):
    date = pd.to_datetime(date)
    year = date.year
    week_number = date.isocalendar().week
    return f"{year}{week_number:02d}"

df['weekno'] = df['연월일'].apply(get_week_of_year)

In [202]:
def format_week_int(week_no):
    # week_no가 202401, 202402 등 정수형이면
    week_no = int(week_no)
    year = week_no // 100
    week = week_no % 100
    return f"{year}{week:02d}"

df_grow['weekno'] = df_grow['week_no'].apply(format_week_int)

In [203]:
# 기 생성된 휴일여부	명절지수	작기정보 칼럼 삭제
df.drop(columns=['휴일여부', '명절지수', '작기정보'], inplace=True)

In [204]:
# 병합하기
merged_df  = pd.merge( df,
                    df_grow[['weekno', 'item_code', 'holiday_flag', 'holiday_score', 'grow_score']],
                    left_on=['weekno', 'item_code'],
                    right_on=['weekno', 'item_code'],
                    how='left'
                )
merged_df.head()

,주차,연월일,품목코드,품목명,품종코드,품종명,등급코드,등급이름,총금액(원),총거래량(kg),...,최고기온,최저기온,평균상대습도,강수량(mm),1시간최고강수량(mm),item_code,weekno,holiday_flag,holiday_score,grow_score
0,2018-01-01~2018-01-07,2018-01-02,5,상추,99,기타상추,12,중,1119400.0,258.0,...,8.3,-2.7,47.0,-9.0,-9.0,1005,201801,3,0.0,21.0
1,2018-01-01~2018-01-07,2018-01-02,5,상추,99,기타상추,13,저,258200.0,164.0,...,6.7,-3.2,74.6,0.0,-9.0,1005,201801,3,0.0,21.0
2,2018-01-01~2018-01-07,2018-01-02,5,상추,99,기타상추,13,저,108000.0,72.0,...,6.7,-3.2,74.6,0.0,-9.0,1005,201801,3,0.0,21.0
3,2018-01-01~2018-01-07,2018-01-02,5,상추,99,기타상추,13,저,152200.0,116.0,...,6.7,-3.2,74.6,0.0,-9.0,1005,201801,3,0.0,21.0
4,2018-01-01~2018-01-07,2018-01-02,5,상추,1,청상추,12,중,60000.0,20.0,...,6.7,-3.2,74.6,0.0,-9.0,1005,201801,3,0.0,21.0


In [205]:
# 강수량, 1시간최고 강수량은 결측치(-9) 혹은 비가 안옴(0)이 많아 0 이하는 0으로 처리
# merged_df['강수량(mm)'] = merged_df['강수량(mm)'<=0].count()
merged_df.loc[merged_df['강수량(mm)']<=0, '강수량(mm)'] = 0
merged_df.loc[merged_df['1시간최고강수량(mm)']<=0, '1시간최고강수량(mm)'] = 0

In [206]:
# 일간 거래 데이터 (필요한 열만 사용)
df_daily_galic = merged_df.loc[:, ['연월일', '품종코드', '등급코드', '총거래량(kg)','주간평균단가(원)','직팜산지코드','일평균기온','최고기온','최저기온','평균상대습도','강수량(mm)','1시간최고강수량(mm)', 'holiday_flag', 'holiday_score','grow_score', '총금액(원)']]
df_daily_galic.head()

,연월일,품종코드,등급코드,총거래량(kg),주간평균단가(원),직팜산지코드,일평균기온,최고기온,최저기온,평균상대습도,강수량(mm),1시간최고강수량(mm),holiday_flag,holiday_score,grow_score,총금액(원)
0,2018-01-02,99,12,258.0,3805.472724,1117,2.7,8.3,-2.7,47.0,0.0,0.0,3,0.0,21.0,1119400.0
1,2018-01-02,99,13,164.0,3805.472724,1102,0.7,6.7,-3.2,74.6,0.0,0.0,3,0.0,21.0,258200.0
2,2018-01-02,99,13,72.0,3805.472724,1103,0.7,6.7,-3.2,74.6,0.0,0.0,3,0.0,21.0,108000.0
3,2018-01-02,99,13,116.0,3805.472724,1102,0.7,6.7,-3.2,74.6,0.0,0.0,3,0.0,21.0,152200.0
4,2018-01-02,1,12,20.0,3805.472724,1107,0.7,6.7,-3.2,74.6,0.0,0.0,3,0.0,21.0,60000.0


In [207]:
# 주간 거래 데이터 병합
df_galic = merged_df.loc[:, ['weekno', '품종코드', '등급코드', '총금액(원)', '총거래량(kg)','직팜산지코드','일평균기온','최고기온','최저기온','평균상대습도','강수량(mm)','1시간최고강수량(mm)', 'holiday_flag', 'holiday_score','grow_score']]
df_weekly_galic = df_galic.groupby(['weekno', '품종코드', '등급코드', '직팜산지코드']).agg({
    '총금액(원)' : 'sum', 
    '총거래량(kg)' : 'sum', 
    '일평균기온': 'mean', 
    '최고기온': 'max', 
    '최저기온' : 'min', 
    '평균상대습도' : 'mean', 
    '강수량(mm)' : 'sum', 
    '1시간최고강수량(mm)' : 'sum', 
    'holiday_flag' : 'sum', 
    'holiday_score' : 'sum', 
    'grow_score' : 'sum'
}).reset_index()

In [208]:
import pandas as pd
from datetime import date  # 여기서 date만 가져옴

# ✅ 주간 단가 계산
df_weekly_galic['평균단가(원)'] = round(df_weekly_galic['총금액(원)'] / df_weekly_galic['총거래량(kg)'])

# ✅ weekno에서 year, week 분리
df_weekly_galic['year'] = df_weekly_galic['weekno'].astype(str).str[:4].astype(int)
df_weekly_galic['week'] = df_weekly_galic['weekno'].astype(str).str[4:].astype(int)

# ✅ 연도별 최대 주차 수
def get_max_week(year):
    return date(year, 12, 28).isocalendar()[1]

# ✅ 유효한 주차만 필터링
df_weekly_galic = df_weekly_galic[df_weekly_galic.apply(
    lambda row: 1 <= row['week'] <= get_max_week(row['year']), axis=1
)].copy()

# ✅ 주차 → 주간 시작일로 변환
def year_week_to_date(year, week):
    return date.fromisocalendar(year, week, 1)  # ← 여기도 date

df_weekly_galic['week_start'] = df_weekly_galic.apply(
    lambda row: year_week_to_date(row['year'], row['week']),
    axis=1
)
df_weekly_galic['week_start'] = pd.to_datetime(df_weekly_galic['week_start'])

# ✅ weekno 제거
df_weekly_galic.drop(columns='weekno', inplace=True)


In [209]:
# # 주간 단가 계산
# df_weekly_galic['평균단가(원)'] = round(df_weekly_galic['총금액(원)'] / df_weekly_galic['총거래량(kg)'])

# # weekno에서 year, week 분리 (문자열 슬라이싱)
# df_weekly_galic['year'] = df_weekly_galic['weekno'].astype(str).str[:4]
# df_weekly_galic['week'] = df_weekly_galic['weekno'].astype(str).str[4:]

# # 주간 시작일 입력 (시계열 특성 - prophet)
# import datetime

# def year_week_to_date(year, week):
#     # ISO 주차는 매년 첫 번째 주의 월요일이 기준
#     return datetime.date.fromisocalendar(int(year), int(week), 1)  # 1: 월요일

# df_weekly_galic['week_start'] = df_weekly_galic.apply(lambda row: year_week_to_date(row['year'], row['week']), axis=1)
# df_weekly_galic['week_start'] = pd.to_datetime(df_weekly_galic['week_start'])
# df_weekly_galic.drop(columns='weekno', inplace=True)

In [210]:
# 저장
# df_daily_galic.to_csv('data/trade_daily_galic.csv', encoding='cp949', index=False)
df_weekly_galic.to_csv('../data/trade_weekly_galic.csv', encoding='cp949', index=False)

# 샘플파일 생성 (chat과 편안한 상담용 )
# df_daily_galic_sample = df_daily_galic.iloc[:100]
# df_daily_galic_sample.to_csv('data/trade_daily_galic_sample.csv', encoding='cp949', index=False)
# df_weekly_galic_sample = df_weekly_galic.iloc[:100]
# df_weekly_galic_sample.to_csv('data/trade_weekly_galic_sample.csv', encoding='cp949', index=False)

In [211]:
df_weekly_galic

,품종코드,등급코드,직팜산지코드,총금액(원),총거래량(kg),일평균기온,최고기온,최저기온,평균상대습도,강수량(mm),1시간최고강수량(mm),holiday_flag,holiday_score,grow_score,평균단가(원),year,week,week_start
0,0,11,1000,2198060.0,404.0,-4.120,2.9,-10.5,43.8200,0.0,0.0,15,0.0,105.0,5441.0,2018,1,2018-01-01
1,0,11,1027,420000.0,84.0,-3.200,0.7,-7.2,50.3000,0.0,0.0,3,0.0,21.0,5000.0,2018,1,2018-01-01
2,0,11,1029,180000.0,36.0,-2.500,1.0,-6.7,69.0000,0.0,0.0,3,0.0,21.0,5000.0,2018,1,2018-01-01
3,0,11,1030,254400.0,48.0,-1.900,2.9,-6.6,72.4000,0.0,0.0,3,0.0,21.0,5300.0,2018,1,2018-01-01
4,0,11,1047,170300.0,26.0,-1.400,0.8,-4.3,46.3000,0.0,0.0,3,0.0,21.0,6550.0,2018,1,2018-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286505,99,13,1120,342500.0,348.0,19.800,28.0,13.2,68.3000,0.9,0.9,8,0.0,84.0,984.0,2025,22,2025-05-26
286506,99,13,1127,6000.0,14.0,18.500,27.6,11.1,73.4500,0.0,0.0,4,0.0,42.0,429.0,2025,22,2025-05-26
286507,99,13,1135,95800.0,96.0,17.800,25.9,11.4,63.0750,0.0,0.0,8,0.0,84.0,998.0,2025,22,2025-05-26
286508,99,13,1142,66000.0,48.0,18.900,26.4,13.7,69.9000,0.0,0.0,2,0.0,21.0,1375.0,2025,22,2025-05-26


In [212]:
df = pd.concat([df_weekly_galic.drop(columns=['평균단가(원)', '총금액(원)']), df_weekly_galic.iloc[:, -4]], axis=1)
# df = pd.concat([df_weekly_galic_sample.drop(columns=['평균단가(원)', '총금액(원)']), df_weekly_galic_sample.iloc[:, -4]], axis=1)

In [213]:
df.to_csv('../data/trade_weekly_galic.csv', encoding='cp949', index=False)

# 모델링

In [214]:
import pandas as pd
import numpy as np

In [215]:
df_weekly_galic = pd.read_csv('../data/trade_weekly_galic.csv', encoding='cp949')
# df_weekly_galic_sample = pd.read_csv('data/trade_weekly_galic_sample.csv', encoding='cp949')

In [216]:
print(df_weekly_galic.isna().sum())  # 대체로 산지가 없거나 수입산인 경우 기후 데이터 없음
df_weekly_galic_drop = df_weekly_galic.dropna()

품종코드                0
등급코드                0
직팜산지코드              0
총거래량(kg)            0
일평균기온            1101
최고기온             1101
최저기온             1101
평균상대습도           1101
강수량(mm)             0
1시간최고강수량(mm)        0
holiday_flag        0
holiday_score       0
grow_score          0
year                0
week                0
week_start          0
평균단가(원)             0
dtype: int64


In [217]:
df_weekly_galic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 286461 entries, 0 to 286460
Data columns (total 17 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   품종코드           286461 non-null  int64  
 1   등급코드           286461 non-null  int64  
 2   직팜산지코드         286461 non-null  int64  
 3   총거래량(kg)       286461 non-null  float64
 4   일평균기온          285360 non-null  float64
 5   최고기온           285360 non-null  float64
 6   최저기온           285360 non-null  float64
 7   평균상대습도         285360 non-null  float64
 8   강수량(mm)        286461 non-null  float64
 9   1시간최고강수량(mm)   286461 non-null  float64
 10  holiday_flag   286461 non-null  int64  
 11  holiday_score  286461 non-null  float64
 12  grow_score     286461 non-null  float64
 13  year           286461 non-null  int64  
 14  week           286461 non-null  int64  
 15  week_start     286461 non-null  object 
 16  평균단가(원)        286461 non-null  float64
dtypes: float64(10), int64(6), obj

In [218]:
# 라이브러리 임포트
import pandas as pd
import numpy as np

In [219]:
# 데이터 로드
df = pd.read_csv('../data/trade_weekly_galic.csv', encoding='cp949', parse_dates=['week_start'])
print(df.shape)
df.head()

(286461, 17)


,품종코드,등급코드,직팜산지코드,총거래량(kg),일평균기온,최고기온,최저기온,평균상대습도,강수량(mm),1시간최고강수량(mm),holiday_flag,holiday_score,grow_score,year,week,week_start,평균단가(원)
0,0,11,1000,404.0,-4.12,2.9,-10.5,43.82,0.0,0.0,15,0.0,105.0,2018,1,2018-01-01,5441.0
1,0,11,1027,84.0,-3.20,0.7,-7.2,50.30,0.0,0.0,3,0.0,21.0,2018,1,2018-01-01,5000.0
2,0,11,1029,36.0,-2.50,1.0,-6.7,69.00,0.0,0.0,3,0.0,21.0,2018,1,2018-01-01,5000.0
3,0,11,1030,48.0,-1.90,2.9,-6.6,72.40,0.0,0.0,3,0.0,21.0,2018,1,2018-01-01,5300.0
4,0,11,1047,26.0,-1.40,0.8,-4.3,46.30,0.0,0.0,3,0.0,21.0,2018,1,2018-01-01,6550.0


In [220]:
# 결측치 처리 (간단히 결측치 행 제거)
df = df.dropna()
print(df.isna().sum())

품종코드             0
등급코드             0
직팜산지코드           0
총거래량(kg)         0
일평균기온            0
최고기온             0
최저기온             0
평균상대습도           0
강수량(mm)          0
1시간최고강수량(mm)     0
holiday_flag     0
holiday_score    0
grow_score       0
year             0
week             0
week_start       0
평균단가(원)          0
dtype: int64


In [221]:
df.head()

,품종코드,등급코드,직팜산지코드,총거래량(kg),일평균기온,최고기온,최저기온,평균상대습도,강수량(mm),1시간최고강수량(mm),holiday_flag,holiday_score,grow_score,year,week,week_start,평균단가(원)
0,0,11,1000,404.0,-4.12,2.9,-10.5,43.82,0.0,0.0,15,0.0,105.0,2018,1,2018-01-01,5441.0
1,0,11,1027,84.0,-3.20,0.7,-7.2,50.30,0.0,0.0,3,0.0,21.0,2018,1,2018-01-01,5000.0
2,0,11,1029,36.0,-2.50,1.0,-6.7,69.00,0.0,0.0,3,0.0,21.0,2018,1,2018-01-01,5000.0
3,0,11,1030,48.0,-1.90,2.9,-6.6,72.40,0.0,0.0,3,0.0,21.0,2018,1,2018-01-01,5300.0
4,0,11,1047,26.0,-1.40,0.8,-4.3,46.30,0.0,0.0,3,0.0,21.0,2018,1,2018-01-01,6550.0


In [222]:
# year, week, week_start 컬럼을 제일 앞으로 이동
front_cols = ['year', 'week', 'week_start']
other_cols = [col for col in df.columns if col not in front_cols]
df = df[front_cols + other_cols]
df.reset_index(drop=True, inplace=True)
df

,year,week,week_start,품종코드,등급코드,직팜산지코드,총거래량(kg),일평균기온,최고기온,최저기온,평균상대습도,강수량(mm),1시간최고강수량(mm),holiday_flag,holiday_score,grow_score,평균단가(원)
0,2018,1,2018-01-01,0,11,1000,404.0,-4.120,2.9,-10.5,43.8200,0.0,0.0,15,0.0,105.0,5441.0
1,2018,1,2018-01-01,0,11,1027,84.0,-3.200,0.7,-7.2,50.3000,0.0,0.0,3,0.0,21.0,5000.0
2,2018,1,2018-01-01,0,11,1029,36.0,-2.500,1.0,-6.7,69.0000,0.0,0.0,3,0.0,21.0,5000.0
3,2018,1,2018-01-01,0,11,1030,48.0,-1.900,2.9,-6.6,72.4000,0.0,0.0,3,0.0,21.0,5300.0
4,2018,1,2018-01-01,0,11,1047,26.0,-1.400,0.8,-4.3,46.3000,0.0,0.0,3,0.0,21.0,6550.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
285355,2025,22,2025-05-26,99,13,1120,348.0,19.800,28.0,13.2,68.3000,0.9,0.9,8,0.0,84.0,984.0
285356,2025,22,2025-05-26,99,13,1127,14.0,18.500,27.6,11.1,73.4500,0.0,0.0,4,0.0,42.0,429.0
285357,2025,22,2025-05-26,99,13,1135,96.0,17.800,25.9,11.4,63.0750,0.0,0.0,8,0.0,84.0,998.0
285358,2025,22,2025-05-26,99,13,1142,48.0,18.900,26.4,13.7,69.9000,0.0,0.0,2,0.0,21.0,1375.0


In [223]:
# 1. 품종 비율 및 누적비율 계산
item_counts = df['품종코드'].value_counts(normalize=True).reset_index()
item_counts.columns = ['품종코드', '비율']
item_counts['누적비율'] = item_counts['비율'].cumsum()

# 2. 누적비율 80% 이하 품종코드만 선택
main_items = item_counts[item_counts['누적비율'] <= 0.8]['품종코드']

# 3. 기타 품종은 숫자 100으로 통합
df['품종코드_통합'] = df['품종코드'].apply(lambda x: x if x in main_items.values else 100)

# 혹시 중복된 '품종코드' 컬럼이 있을 수 있으니 정리
df = df.loc[:, ~df.columns.duplicated()]

# 품종코드 값을 통합된 값으로 덮어쓰기
df['품종코드'] = df['품종코드_통합']

# 통합 컬럼 삭제
df.drop(columns=['품종코드_통합'], inplace=True)




In [224]:
df['품종코드'].unique()

array([100,   1,   2,   4,   5,  99], dtype=int64)

In [225]:
import pandas as pd

# 1. 주차 기준 키 생성
df['yearweek'] = df['year'] * 100 + df['week']

# 2. 기후 컬럼 정의
climate_cols = [
    '일평균기온', '최고기온', '최저기온',
    '평균상대습도', '강수량(mm)', '1시간최고강수량(mm)'
]

# 3. 등급 포함 지연기후용 테이블 생성
climate_df = (
    df[['직팜산지코드', '등급코드', 'year', 'week'] + climate_cols]
    .drop_duplicates(subset=['직팜산지코드', '등급코드', 'year', 'week'])
    .copy()
)
climate_df['yearweek'] = climate_df['year'] * 100 + climate_df['week']

# 4. 지연 변수 생성 (등급 포함 그룹핑)
climate_df.sort_values(['직팜산지코드', '등급코드', 'yearweek'], inplace=True)

# ✅ 1달~3달 전 주차 평균(t-1 ~ t-3)을 계산
for offset, lag in zip([4, 8, 12], [1, 2, 3]):
    for col in climate_cols:
        climate_df[f'{col}_t-{lag}'] = (
            climate_df
            .groupby(['직팜산지코드', '등급코드'])[col]
            .transform(lambda x: x.shift(offset - 3).rolling(window=4).mean())
        )


# 5. 지연기후만 남기기
climate_df = climate_df.drop(columns=climate_cols)

# 6. 병합 대상: 원본 df 전체
df['yearweek'] = df['year'] * 100 + df['week']
climate_df['yearweek'] = climate_df['yearweek'].astype(int)

df['yearweek'] = df['yearweek'].astype('int64')
climate_df['yearweek'] = climate_df['yearweek'].astype('int64')

# 7. 병합을 위한 키: 직팜산지코드 + 등급코드 + yearweek
df_final = pd.merge(
    df,
    climate_df,
    how='left',
    on=['직팜산지코드', '등급코드', 'yearweek']
)

# 8. yearweek 제거
df_final.drop(columns='yearweek', inplace=True)

# 9. 컬럼 정리 (필요 시)
df_final.rename(columns={
    'year_x': 'year',
    'week_x': 'week'
}, inplace=True)

df_final.drop(columns=['year_y', 'week_y'], errors='ignore', inplace=True)

# 10. 결과 확인
print("✅ 등급 포함 지연기후 병합 완료")
print("전체 행:", len(df_final))
print("지연기후 샘플:\n", df_final[[col for col in df_final.columns if '_t-' in col]].head())


✅ 등급 포함 지연기후 병합 완료
전체 행: 285360
지연기후 샘플:
    일평균기온_t-1  최고기온_t-1  최저기온_t-1  평균상대습도_t-1  강수량(mm)_t-1  1시간최고강수량(mm)_t-1  \
0        NaN       NaN       NaN         NaN          NaN               NaN   
1        NaN       NaN       NaN         NaN          NaN               NaN   
2        NaN       NaN       NaN         NaN          NaN               NaN   
3        NaN       NaN       NaN         NaN          NaN               NaN   
4        NaN       NaN       NaN         NaN          NaN               NaN   

   일평균기온_t-2  최고기온_t-2  최저기온_t-2  평균상대습도_t-2  강수량(mm)_t-2  1시간최고강수량(mm)_t-2  \
0        NaN       NaN       NaN         NaN          NaN               NaN   
1        NaN       NaN       NaN         NaN          NaN               NaN   
2        NaN       NaN       NaN         NaN          NaN               NaN   
3        NaN       NaN       NaN         NaN          NaN               NaN   
4        NaN       NaN       NaN         NaN          NaN               NaN   

   일평균기온

In [226]:
df_final.rename(columns={
    'year_x': 'year',
    'week_x': 'week'
}, inplace=True)

# 확인
df_final[['year', 'week', '직팜산지코드'] + [col for col in df_final.columns if '_t-' in col]].head()



,year,week,직팜산지코드,일평균기온_t-1,최고기온_t-1,최저기온_t-1,평균상대습도_t-1,강수량(mm)_t-1,1시간최고강수량(mm)_t-1,일평균기온_t-2,...,최저기온_t-2,평균상대습도_t-2,강수량(mm)_t-2,1시간최고강수량(mm)_t-2,일평균기온_t-3,최고기온_t-3,최저기온_t-3,평균상대습도_t-3,강수량(mm)_t-3,1시간최고강수량(mm)_t-3
0,2018,1,1000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2018,1,1027,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2018,1,1029,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2018,1,1030,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2018,1,1047,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [227]:
# 1. NaN 제거
df_final.dropna(inplace=True)

# 2. 총거래량이 0 이하인 행 제거
df_final = df_final[df_final['총거래량(kg)'] > 0]

# 3. 결과 확인
print("✅ 결측치 및 거래량 0 제거 완료")
print("남은 행 수:", len(df_final))


✅ 결측치 및 거래량 0 제거 완료
남은 행 수: 272381


In [228]:
# 7. CSV 저장

df_final.to_csv(f"../data/{crop_name}_월차낼게요.csv", index=False, encoding='cp949')


print(f"✅ 저장 완료")

✅ 저장 완료


# 정확도 검증

In [229]:
# df_final = pd.merge(
#     df,
#     climate_df,
#     how='left',
#     on=['직팜산지코드', '등급코드', 'yearweek']
# )


In [230]:
# def validate_lag_mean_strict(df_original, df_lagged, target_col, lag_index):
#     results = []
#     label = f'{target_col}_t-{lag_index}'
#     offset = lag_index * 4
#     shift_start = offset - 3

#     df_base = (
#         df_original[['직팜산지코드', '등급코드', 'year', 'week', 'yearweek', target_col]]
#         .drop_duplicates(subset=['직팜산지코드', '등급코드', 'year', 'week'])
#         .copy()
#     )
#     df_base.sort_values(['직팜산지코드', '등급코드', 'yearweek'], inplace=True)

#     for keys, group in df_base.groupby(['직팜산지코드', '등급코드']):
#         group = group.reset_index(drop=True)
#         rolling = group[[target_col]].shift(shift_start).rolling(window=4).mean()
#         group['수동_계산'] = rolling.reset_index(drop=True)

#         merged = pd.merge(
#             group,
#             df_lagged[['직팜산지코드', '등급코드', 'yearweek', label]],
#             on=['직팜산지코드', '등급코드', 'yearweek'],
#             how='left'
#         ).rename(columns={label: '자동_생성'})

#         merged['일치'] = merged['수동_계산'].round(3) == merged['자동_생성'].round(3)
#         results.append(merged)

#     result_df = pd.concat(results, ignore_index=True)
#     valid = result_df.dropna(subset=['수동_계산', '자동_생성'])
#     accuracy = (valid['수동_계산'].round(3) == valid['자동_생성'].round(3)).mean()
#     return result_df, accuracy


In [231]:
# df['yearweek'] = df['year'] * 100 + df['week']  # 혹시 없으면 만들기

# df_check_t1, acc_t1 = validate_lag_mean_strict(df, df_final, '일평균기온', 1)
# df_check_t2, acc_t2 = validate_lag_mean_strict(df, df_final, '일평균기온', 2)
# df_check_t3, acc_t3 = validate_lag_mean_strict(df, df_final, '일평균기온', 3)

# print(f"✅ t-1 정확도: {acc_t1*100:.2f}%")
# print(f"✅ t-2 정확도: {acc_t2*100:.2f}%")
# print(f"✅ t-3 정확도: {acc_t3*100:.2f}%")

# df_check_t1[['year', 'week', '일평균기온', '수동_계산', '자동_생성', '일치']].tail(10)
